# Monthly Revenue Trend

In [0]:
SELECT 
    d.year,
    d.month,
    SUM(f.revenue_usd) AS total_revenue
FROM retailer.gold.fact_sales f
JOIN retailer.gold.dim_date d
    ON f.order_date = d.date
WHERE d.year = 2019
GROUP BY d.year, d.month
ORDER BY d.month;

# Peak Month Analysis (Top 3 + %)

In [0]:
WITH monthly AS (
    SELECT 
        d.month,
        SUM(f.revenue_usd) AS revenue
    FROM retailer.gold.fact_sales f
    JOIN retailer.gold.dim_date d
        ON f.order_date = d.date
    WHERE d.year = 2019
    GROUP BY d.month
),
total AS (
    SELECT SUM(revenue) AS total_revenue FROM monthly
)
SELECT 
    m.month,
    m.revenue,
    ROUND((m.revenue / t.total_revenue) * 100, 2) AS pct_of_total
FROM monthly m, total t
ORDER BY m.revenue DESC
LIMIT 3;

# Holiday Drivers

In [0]:
WITH monthly AS (
    SELECT 
        d.month,
        SUM(f.revenue_usd) AS revenue
    FROM retailer.gold.fact_sales f
    JOIN retailer.gold.dim_date d
        ON f.order_date = d.date
    WHERE d.year = 2019 AND d.month IN (4,5,6)
    GROUP BY d.month
),
top_months AS (
    SELECT month
    FROM monthly
    ORDER BY revenue DESC
    LIMIT 2
)
SELECT 
    p.category,
    SUM(f.revenue_usd) AS revenue
FROM retailer.gold.fact_sales f
JOIN retailer.gold.dim_products p
    ON f.product_key = p.product_key
JOIN retailer.gold.dim_date d
    ON f.order_date = d.date
WHERE d.month IN (SELECT month FROM top_months)
GROUP BY p.category
ORDER BY revenue DESC
LIMIT 3;

# Delivery Performance

In [0]:
SELECT 
    AVG(DATEDIFF(delivery_date, order_date)) AS avg_delivery_days
FROM retailer.gold.fact_sales;

# Country Delivery Issues (Slowest 5)

In [0]:
SELECT 
    s.country,
    AVG(DATEDIFF(f.delivery_date, f.order_date)) AS avg_delivery_days
FROM retailer.gold.fact_sales f
JOIN retailer.gold.dim_stores s
    ON f.store_key = s.store_key
GROUP BY s.country
ORDER BY avg_delivery_days DESC
LIMIT 5;

# Channel Performance

In [0]:
SELECT 
    c.continent,
    CASE 
        WHEN f.store_key IS NULL THEN 'Online'
        ELSE 'In-Store'
    END AS channel,
    SUM(f.revenue_usd) / COUNT(DISTINCT f.order_number) AS AOV
FROM retailer.gold.fact_sales f
JOIN retailer.gold.dim_customers c
    ON f.customer_key = c.customer_key
GROUP BY c.continent, channel;

# Volume Leaders

In [0]:
SELECT 
    p.category,
    SUM(f.quantity) AS total_units
FROM retailer.gold.fact_sales f
JOIN retailer.gold.dim_products p
    ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY total_units DESC
LIMIT 5;

# Revenue Leaders

In [0]:
SELECT 
    p.category,
    SUM(f.revenue_usd) AS total_revenue
FROM retailer.gold.fact_sales f
JOIN retailer.gold.dim_products p
    ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY total_revenue DESC
LIMIT 5;

# Customer Profile

In [0]:
SELECT 
    c.continent,
    c.gender,
    COUNT(DISTINCT c.customer_key) AS customer_count,
    SUM(f.revenue_usd) AS total_spending
FROM retailer.gold.fact_sales f
JOIN retailer.gold.dim_customers c
    ON f.customer_key = c.customer_key
GROUP BY c.continent, c.gender
ORDER BY total_spending DESC;

# Customer Loyalty

In [0]:
WITH customer_orders AS (
    SELECT 
        customer_key,
        COUNT(DISTINCT order_number) AS order_count
    FROM retailer.gold.fact_sales
    GROUP BY customer_key
),
repeat_customers AS (
    SELECT customer_key
    FROM customer_orders
    WHERE order_count >= 2
)
SELECT 
    c.continent,
    COUNT(DISTINCT r.customer_key) * 100.0 / 
    COUNT(DISTINCT c.customer_key) AS repeat_rate_pct
FROM retailer.gold.dim_customers c
LEFT JOIN repeat_customers r
    ON c.customer_key = r.customer_key
GROUP BY c.continent;